# Task 4 — watch Nanochat inference and its KV cache

This is an educational notebook, not a training run. It loads the exact checkpoint and tokenizer selected in task4/config.json, formats one chat prompt, observes Nanochat's forward calls, and compares a cached prefill with an uncached forward pass. No checkpoint or result file is overwritten. Use the project's .venv as your notebook kernel.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys
import torch

# Find the repository whether Jupyter starts at the repository root or task4/.
starting_here = Path.cwd().resolve()
repo_root = next(
    (path for path in (starting_here, *starting_here.parents)
     if (path / "task4" / "common.py").is_file()),
    None,
)
assert repo_root is not None, "Open this notebook from the nanochat repository."
sys.path.insert(0, str(repo_root / "task4"))
import common

# The notebook uses the same model selection and checks as the four CLI scripts.
options = SimpleNamespace(
    config=str(repo_root / "task4" / "config.json"),
    checkpoint_dir=None, tokenizer_dir=None, step=None, stage=None,
    device=None, dtype=None, cache_dir=None, output_root=None,
)
config, spec, generation, benchmark = common.configuration(options)
identity = common.verify_files(spec)
model, tokenizer, normal_engine = common.load_nanochat(spec, identity)

print("Stage:", identity["stage"])
print("Step:", identity["step"])
print("Device:", identity["device"])
print("Layers:", model.config.n_layer)
print("Training context:", model.config.sequence_len)
print("Vocabulary:", tokenizer.get_vocab_size())
print("Parameters:", identity["trainable_parameters"])

## 1. Build the chat prompt

The CLI does not feed plain user text directly to the transformer. It prepends BOS, wraps the user message in special tokens, and finishes with assistant-start. Each item in the resulting list is a token ID. The model's next prediction is what follows assistant-start.

In [ ]:
PROMPT = "Which planet is known as the Red Planet?"
prompt_ids = common.chat_prefix(tokenizer, PROMPT)
print("User text:", PROMPT)
print("Token IDs:", prompt_ids)
print("Prompt length:", len(prompt_ids), "tokens")
print("Special token IDs:")
for name in ("<|bos|>", "<|user_start|>", "<|user_end|>", "<|assistant_start|>"):
    print(f"  {name:22} -> {tokenizer.encode_special(name)}")
print("Readable pieces:")
for position, token_id in enumerate(prompt_ids):
    print(f"{position:>2}: id={token_id:>5} piece={tokenizer.decode([token_id])!r}")

## 2. Observe prefill and decoding

Nanochat's Engine first calls the transformer once with all prompt tokens; that is prefill. It caches keys and values produced at those positions. As the assistant generates, later calls send one new token at a time. The recorder below *observes* calls; it does not modify the model's calculations.

In [ ]:
from nanochat.engine import Engine

class ForwardRecorder:
    def __init__(self, original_model):
        self.original_model = original_model
        self.config = original_model.config
        self.calls = []

    def get_device(self):
        return self.original_model.get_device()

    def forward(self, ids, kv_cache=None):
        before = None if kv_cache is None else kv_cache.get_pos()
        logits = self.original_model.forward(ids, kv_cache=kv_cache)
        after = None if kv_cache is None else kv_cache.get_pos()
        self.calls.append({
            "phase": "prefill" if not self.calls else "decode",
            "input_shape": tuple(ids.shape),
            "cache_position_before": before,
            "cache_position_after": after,
        })
        return logits

recorder = ForwardRecorder(model)
traced_engine = Engine(recorder, tokenizer)
MAX_NEW_TOKENS = 8
common.context_check(model, prompt_ids, MAX_NEW_TOKENS)
answer = common.generate_reply(
    traced_engine, tokenizer, prompt_ids,
    temperature=0.7, top_k=50, max_new_tokens=MAX_NEW_TOKENS, seed=42,
)
print("Generated reply:", repr(answer["response"]))
print("Generated token IDs:", answer["generated_ids"])
print("Stop reason:", answer["stop_reason"])
print("\nTransformer forward calls:")
for number, call in enumerate(recorder.calls):
    print(number, call)

Notice the first input shape is (1, prompt_length). Later shapes are (1, 1). The cache position increases; the model does not recompute all earlier token keys and values at every decoding step. Nanochat's current Engine also performs a forward call after the last allowed generated token, although that final set of logits is not used if the token budget is exhausted.

## 3. Check the cached prefill against a normal forward pass

A cache should make inference faster without changing what the prompt predicts. This cell compares the last-position vocabulary logits from (a) one ordinary forward pass and (b) one forward pass that writes keys and values into a fresh cache. Tiny numerical differences can occur on different devices.

In [ ]:
from nanochat.engine import KVCache
from nanochat.common import COMPUTE_DTYPE

device = model.get_device()
input_tensor = torch.tensor([prompt_ids], dtype=torch.long, device=device)
head_dim = model.config.n_embd // model.config.n_head
fresh_cache = KVCache(
    batch_size=1,
    num_heads=model.config.n_kv_head,
    seq_len=len(prompt_ids),
    head_dim=head_dim,
    num_layers=model.config.n_layer,
    device=device,
    dtype=COMPUTE_DTYPE,
)
with torch.inference_mode():
    ordinary_logits = model(input_tensor)[0, -1]
    cached_logits = model(input_tensor, kv_cache=fresh_cache)[0, -1]

largest_difference = (ordinary_logits - cached_logits).abs().max().item()
print("Logits per position:", ordinary_logits.numel())
print("Cache now holds positions:", fresh_cache.get_pos())
print("Largest logit difference:", largest_difference)
print("Top ordinary token ID:", ordinary_logits.argmax().item())
print("Top cached token ID:", cached_logits.argmax().item())

## 4. Inspect what the cache stores

The tensors below hold a key and a value for each layer and prompt position. Their axes are (layers, batch, allocated positions, KV heads, head width). Queries are produced for the *current* positions; they are not stored in this KV cache. This cell shows shapes and memory, not thousands of raw vector numbers.

In [ ]:
key_tensor = fresh_cache.k_cache
value_tensor = fresh_cache.v_cache
print("Key shape:", tuple(key_tensor.shape))
print("Value shape:", tuple(value_tensor.shape))
print("Prompt positions filled:", fresh_cache.get_pos())
print("Key bytes:", key_tensor.numel() * key_tensor.element_size())
print("Value bytes:", value_tensor.numel() * value_tensor.element_size())
print("First layer, first token, first head: first eight key values:")
print(key_tensor[0, 0, 0, 0, :8].tolist())
print("First layer, first token, first head: first eight value values:")
print(value_tensor[0, 0, 0, 0, :8].tolist())

## 5. What happens on the next chat turn?

The CLI keeps the old conversation *as token IDs*. But Nanochat's Engine starts a **new cache** when called again. It prefills that new cache with the accumulated conversation; it does not carry the previous call's cache across turns. The model remembers because the earlier tokens are included again.

In [ ]:
assistant_end = tokenizer.encode_special("<|assistant_end|>")
history = prompt_ids + answer["generated_ids"]
if history[-1] != assistant_end:
    history.append(assistant_end)
next_prompt_ids = common.chat_prefix(
    tokenizer, "Can you say that in fewer words?", history=history
)
print("First-turn prompt length:", len(prompt_ids))
print("Second-turn prompt length including history:", len(next_prompt_ids))

second_recorder = ForwardRecorder(model)
second_engine = Engine(second_recorder, tokenizer)
common.context_check(model, next_prompt_ids, 3)
second_answer = common.generate_reply(
    second_engine, tokenizer, next_prompt_ids,
    temperature=0.7, top_k=50, max_new_tokens=3, seed=43,
)
print("Second reply:", repr(second_answer["response"]))
print("First call on turn 2:", second_recorder.calls[0])
print("That prefill starts at cache position 0 and includes old + new tokens.")

## Try it yourself

Change PROMPT, MAX_NEW_TOKENS, temperature, or the second user message and rerun the relevant cells. Questions to answer in your own words: Why does turn 2's prefill grow? Which forward calls have shape (1, 1)? What is stored in the cache? How is the cached result different from the no-cache logits? For the Task 4 report, describe your observations rather than pasting this notebook's prose.